# Generate and prepare blind comparisons

Attach the supplied project zip and the saved training notebook outputs. Enable GPU and internet. Inspect the sanity outputs before running full generation. This notebook has not been run on Kaggle yet.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
REPO = Path('/kaggle/working/small-model-alignment')
if not REPO.exists():
    archives = list(Path('/kaggle/input').rglob('small-model-alignment.zip'))
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall('/kaggle/working')
    else:
        sources = [p.parent for p in Path('/kaggle/input').rglob('submission_adapter.py')
                   if (p.parent / 'alignment/core.py').is_file()
                   and (p.parent / 'configs/sft.json').is_file()
                   and not (p.parent / 'outputs/sft/complete.json').exists()]
        if len(sources) != 1:
            raise RuntimeError('Attach exactly one project dataset containing the ZIP or unpacked project files')
        shutil.copytree(sources[0], REPO)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
def run(script, *arguments):
    subprocess.run([sys.executable, script, *map(str, arguments)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-persona.txt'], check=True)

In [ ]:
from alignment.io import read_rows
candidates = [p.parent.parent.parent for p in Path('/kaggle/input').rglob('complete.json') if p.parent.name == 'sft' and p.parent.parent.name == 'outputs']
if len(candidates) != 1:
    raise RuntimeError(f'Attach exactly one completed training output dataset; found {candidates}')
TRAINING_ROOT = candidates[0]
print('Training artifacts:', TRAINING_ROOT)
alpaca = TRAINING_ROOT / 'data/prepared/alpaca_eval.json'
hh = REPO / 'eval_sets/hh_rlhf_eval_300_seed42.json'
persona = REPO / 'eval_sets/persona_eval_seed42.json'
jobs = [
    ('qwen-base', 'Qwen/Qwen2.5-0.5B', alpaca, 'base_alpaca', []),
    ('sft', TRAINING_ROOT / 'outputs/sft/best', alpaca, 'sft_alpaca', []),
    ('sft', TRAINING_ROOT / 'outputs/sft/best', hh, 'sft_hh', []),
    ('dpo-uf', TRAINING_ROOT / 'outputs/dpo_uf/best', alpaca, 'dpo_uf', []),
    ('dpo-hh', TRAINING_ROOT / 'outputs/dpo_hh/best', hh, 'dpo_hh', []),
    ('smollm-instruct', 'HuggingFaceTB/SmolLM2-360M-Instruct', persona, 'persona_base', ['--chat-template']),
    ('persona-lora', 'HuggingFaceTB/SmolLM2-360M-Instruct', persona, 'persona_lora', ['--chat-template', '--adapter', TRAINING_ROOT / 'outputs/persona/best']),
]
def generate(job, sanity=False):
    label, model, prompts, filename, extra = job
    output = REPO / f"outputs/eval/{filename}{'_sanity' if sanity else ''}.json"
    run('scripts/generate.py', '--model', model, '--prompts', prompts, '--output', output,
        '--generator', label, *extra, *(['--limit', '3'] if sanity else []))
    return output


## Sanity check

Inspect all three responses per model for prompt formatting, EOS stopping, coherence, and output length. If changing decoding settings, apply the same change to both models in each comparison and use new output paths.

In [ ]:
for job in jobs:
    path = generate(job, sanity=True)
    print(job[0], job[3])
    for row in read_rows(path):
        print(row['instruction'], '\nANSWER:', row['output'], '\n')

## Full generation

Run this cell after inspecting the sanity results. Completed rows are reused only when metadata matches.

In [ ]:
for job in jobs:
    generate(job)

## Pairwise files

Response positions are shuffled with seed 42. Model identity mapping remains in file metadata; judge-facing prompts contain only the instruction and Response A/B.

In [ ]:
for baseline, candidate, filename in [('sft_alpaca', 'dpo_uf', 'uf_pairs'), ('sft_hh', 'dpo_hh', 'hh_pairs'), ('persona_base', 'persona_lora', 'persona_pairs')]:
    output = Path(f'outputs/eval/{filename}.jsonl')
    if not output.exists():
        run('scripts/build_pairs.py', '--baseline', f'outputs/eval/{baseline}.json', '--candidate', f'outputs/eval/{candidate}.json', '--output', output)

## Preserve results

Save a notebook version. Attach these generation outputs, including pairwise files and metadata, to the utility benchmark evaluation notebook.